# Data Understanding

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from utils import parse_number
from config import (
  RAW_REGENCIES_CSV,
  RAW_PROVINCES_CSV,
  UNDERSTANDING_PROVINCES_CSV,
  UNDERSTANDING_REGENCIES_CSV,
  FEATURE_EVALUATION_JSON,
  PROVINCE_COLUMN_MAPPING,
  REGENCY_COLUMN_MAPPING
)

In [ ]:
# Memuat dataset hasil scraping mentah
df_prov = pd.read_csv(RAW_PROVINCES_CSV, dtype=str)
df_reg = pd.read_csv(RAW_REGENCIES_CSV, dtype=str)

# Pemetaan nama kolom ke format standar
df_prov = df_prov.rename(columns=PROVINCE_COLUMN_MAPPING)
df_reg = df_reg.rename(columns=REGENCY_COLUMN_MAPPING)

In [ ]:
PROVINCES_IGNORED_FORMATTED_COLS = ('no', 'province_name')
REGENCIES_IGNORED_FORMATTED_COLS = ('province_id', 'regency_no', 'regency_name')

for col in df_prov.columns:
  if col not in PROVINCES_IGNORED_FORMATTED_COLS:
    df_prov[col] = df_prov[col].apply(parse_number)

for col in df_reg.columns:
  if col not in REGENCIES_IGNORED_FORMATTED_COLS:
    df_reg[col] = df_reg[col].apply(parse_number)

In [ ]:
df_prov.to_csv(UNDERSTANDING_PROVINCES_CSV, index=False)
df_reg.to_csv(UNDERSTANDING_REGENCIES_CSV, index=False)

In [ ]:
print(df_prov.head().to_markdown(index=False))

In [ ]:
print(df_reg.head().to_markdown(index=False))

## Statistik Deskriptif

In [ ]:
df_stats = df_reg.describe().T
print(df_stats.to_markdown())

In [ ]:
total_koperasi = int(df_prov['total_koperasi'].sum())
nib_sum = int(df_prov['koperasi_nib'].sum())
npwp_sum = int(df_prov['koperasi_npwp'].sum())
rat_sum = int(df_prov['koperasi_rat'].sum())
pct_nib = round(nib_sum / total_koperasi * 100, 2) if total_koperasi > 0 else 0.0
pct_npwp = round(npwp_sum / total_koperasi * 100, 2) if total_koperasi > 0 else 0.0
pct_rat = round(rat_sum / total_koperasi * 100, 2) if total_koperasi > 0 else 0.0
simpanan_pokok = float(df_prov['simpanan_pokok'].sum())
simpanan_wajib = float(df_prov['simpanan_wajib'].sum())
nilai_transaksi = float(df_prov['nilai_transaksi'].sum())

print(f"Total Kabupaten/Kota       : {len(df_reg)}")
print(f"Total Koperasi Terdata     : {total_koperasi:,} unit")
print(f"Koperasi Memiliki NIB      : {nib_sum:,} ({pct_nib}%)")
print(f"Koperasi Memiliki NPWP     : {npwp_sum:,} ({pct_npwp}%)")
print(f"Koperasi Telah RAT         : {rat_sum:,} ({pct_rat}%)")
print(f"Akumulasi Simpanan Pokok   : Rp {simpanan_pokok:,.2f}")
print(f"Akumulasi Simpanan Wajib   : Rp {simpanan_wajib:,.2f}")
print(f"Total Nilai Transaksi      : Rp {nilai_transaksi:,.2f}")

## Eksplorasi Data

### Heatmap Korelasi Pearson

In [ ]:
corr_matrix = df_reg.select_dtypes('number').corr()

plt.figure(figsize=(8, 6.5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Matriks Korelasi Pearson')
plt.tight_layout()
plt.show()

### Barplot Top Provinsi Jumlah Koperasi

In [ ]:
TOP_PROVINCES_LIMIT = 10

top_prov = df_prov.sort_values(by='total_koperasi', ascending=False).head(TOP_PROVINCES_LIMIT)

plt.figure(figsize=(10, 5))
sns.barplot(x='total_koperasi', y='province_name', data=top_prov)
plt.title('Provinsi dengan Jumlah Koperasi Terbanyak')
plt.tight_layout()
plt.show()

### Barplot Top Kabupaten/Kota Nilai Transaksi

In [ ]:
TOP_REGENCIES_LIMIT = 15

top_reg = df_reg.sort_values(by='nilai_transaksi', ascending=False).head(TOP_REGENCIES_LIMIT).copy()
top_reg['nilai_juta'] = top_reg['nilai_transaksi'] / 1e6

plt.figure(figsize=(10, 5))
sns.barplot(x='nilai_juta', y='regency_name', data=top_reg)
plt.title('Kabupaten/Kota dengan Nilai Transaksi Tertinggi (Juta Rp)')
plt.tight_layout()
plt.show()

### Distribusi Fitur

In [ ]:
df_dist = df_reg.select_dtypes('number')

fig, axes = plt.subplots(int(np.ceil(df_dist.shape[1] / 4)), 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(df_dist.columns):
    sns.histplot(df_dist[col], kde=True, ax=axes[idx])
    axes[idx].set_title(col)

for ax in axes[df_dist.shape[1]:]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()

### Analisis Variabilitas & Koefisien Variasi

In [ ]:
stats = df_reg.select_dtypes('number').agg(['mean', 'std', 'var']).T
stats['CV (%)'] = (stats['std'] / stats['mean'] * 100)

variability_df = stats.reset_index().rename(columns={'index': 'Fitur'})

print(variability_df.to_markdown(index=False))

### Uji Multikolinearitas (VIF)

In [ ]:
df_vif = df_reg.select_dtypes('number').fillna(0)

X_scaled = StandardScaler().fit_transform(df_vif)
inv_corr = np.linalg.inv(np.corrcoef(X_scaled, rowvar=False))

vif_df = pd.DataFrame({
  'Fitur': df_vif.columns,
  'VIF': np.diag(inv_corr)
})

print(vif_df.to_markdown(index=False))

### Penyimpanan Hasil Uji Statistik

In [ ]:
feature_cols = df_reg.select_dtypes('number').columns.difference(REGENCIES_IGNORED_FORMATTED_COLS).tolist()

evaluation_results = {
  "feature_columns": feature_cols,
  "vif": vif_df.set_index('Fitur')['VIF'].to_dict(),
  "variability_cv": variability_df.set_index('Fitur')['CV (%)'].to_dict(),
  "skewness": df_reg[feature_cols].skew().to_dict()
}

with open(FEATURE_EVALUATION_JSON, 'w', encoding='utf-8') as f:
  json.dump(evaluation_results, f, indent=2)

print(f"Hasil uji statistik fitur berhasil disimpan di : {FEATURE_EVALUATION_JSON}")

## Verifikasi Kualitas Data

### Mengecek Nilai Hilang

In [ ]:
df_mv = df_reg.isnull().sum()

print(df_mv.to_markdown())

### Mengecek Outlier

In [ ]:
df_num = df_reg.select_dtypes('number')

q1 = df_num.quantile(0.25)
q3 = df_num.quantile(0.75)
iqr = q3 - q1
lower = (q1 - 1.5 * iqr).clip(lower=0)
upper = q3 + 1.5 * iqr

out_cnt = ((df_num < lower) | (df_num > upper)).sum()

outliers_df = pd.DataFrame({
  'Fitur': df_num.columns,
  'Skewness': df_num.skew(),
  'Batas Bawah': lower,
  'Batas Atas': upper,
  'Outliers': out_cnt,
  'Persentase (%)': (out_cnt / len(df_num) * 100)
})

print(outliers_df.to_markdown(index=False))